# 20 · Composing functions (Khandelwal & Pavlick, arXiv:2510.01685)

Same question as notebook `19`, new dataset: does GoalScope's goal-head read-out verbalize the
**intermediate value `Fx`** in a two-hop compositional query `y = g(f(x))`
([paper](https://arxiv.org/abs/2510.01685), [dataset](https://huggingface.co/datasets/apoorvkh/composing-functions))?

This notebook is `19` almost verbatim — same imports, same goal-head loading, same
`_chat`/`GOALSCOPE_PROMPTS` registry (+ one new `intermediate_value` entry), same `run_case`. Only
the data-setup section (loading composing-functions instead of FV tasks) is new.

In [2]:
%load_ext autoreload
%autoreload 2

import os, sys, json, random, logging
from pathlib import Path
from glob import glob

# pick a free GPU (32B fits on one A100-80GB) -- check `nvidia-smi` first
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append("../..")

In [3]:
import torch
from rich.console import Console
from rich.theme import Theme
from rich.markup import escape

from src.utils import logging_utils
console = Console(theme=Theme({"default": "white"}), highlight=False)
def rich_print(*a, **k): console.print(*a, **k)
logging_utils.configure(level=logging.INFO)
logger = logging.getLogger(__name__)

In [4]:
from src.models import ModelandTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"
mt = ModelandTokenizer(
    model_key=MODEL_NAME, dtype=torch.bfloat16,
    device_map="auto", attn_implementation="eager",
)
print(f"n_layer={mt.n_layer}, n_embd={mt.n_embd}, n_heads={mt.model.config.num_attention_heads}")

2026-09-17 16:03:59 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in models
If not found in cache, model will be downloaded from HuggingFace to cache directory: None
2026-09-17 16:03:59 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-17 16:03:59 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/config.json?%2FQwen%2FQwen2.5-Coder-32B-Instruct%2Fresolve%2Fmain%2Fconfig.json=&etag=%22f51f25852c87ac80dc2fbd2ac13ffa285dba77c7%22 "HTTP/1.1 200 OK"
2026-09-17 16:03:59 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


2026-09-17 16:03:59 huggingface_hub.utils._http WARNING  Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-09-17 16:03:59 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/config.json?%2FQwen%2FQwen2.5-Coder-32B-Instruct%2Fresolve%2Fmain%2Fconfig.json=&etag=%22f51f25852c87ac80dc2fbd2ac13ffa285dba77c7%22 "HTTP/1.1 200 OK"
2026-09-17 16:03:59 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-17 16:03:59 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/tokenizer_config.json?%2FQwen%2FQwen2.5-Coder-32B-Instruct%2Fresolve%2Fmain%2Ftokenizer_config.json=&etag=%22acee076f49bf3c029

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-09-17 16:04:24 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-17 16:04:24 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/generation_config.json?%2FQwen%2FQwen2.5-Coder-32B-Instruct%2Fresolve%2Fmain%2Fgeneration_config.json=&etag=%22c28f9c697cbc09f047434efec57556505af31111%22 "HTTP/1.1 200 OK"
2026-09-17 16:04:24 src.models INFO     loaded model <Qwen/Qwen2.5-Coder-32B-Instruct> | size: 62492.135 MB | dtype: torch.bfloat16 | device: cuda:0
n_layer=64, n_embd=5120, n_heads=40


## Goal heads

Load the frozen goal-head set (git-tracked `head_sets/<model>/goal__lam20_thr0.9_band*.json`, same
set the goalscope evals use). Falls back to the trained mask checkpoint if no JSON is present.

In [5]:
slug = MODEL_NAME.split("/")[-1]
_json = sorted(glob(f"../../head_sets/{slug}/goal__lam20_thr0.9_band*.json"))
if _json:
    goal_heads = [tuple(x) for x in json.load(open(_json[0]))]
    print(f"goal heads from {Path(_json[0]).name}: {len(goal_heads)}")
else:  # fall back to the mask checkpoint (see __non_code.ipynb)
    from src.goals.head_loc_attn import HeadMask
    from src.utils import env_utils
    ckpt = torch.load(os.path.join(env_utils.DEFAULT_RESULTS_DIR,
                      f"localization/{slug}/lam20/head_mask.pt"), map_location="cpu")
    LAYERS = ckpt["layers"]
    mask = HeadMask(len(LAYERS), ckpt["mask_values"].shape[1])
    with torch.no_grad():
        mask.values.copy_(ckpt["mask_values"])
        band = {"Qwen2.5-Coder-32B-Instruct": (25, 56)}.get(slug)
        if band:
            mask.values[:band[0]] = 0; mask.values[band[1]:] = 0
    goal_heads = mask.selected_heads(LAYERS, threshold=0.9)
    print(f"goal heads from mask ckpt: {len(goal_heads)}")

goal heads from goal__lam20_thr0.9_band25-56.json: 122


## Read-out prompt registry — *choose the GoalScope setup*

`apply_goalscope(..., goalscope_prompt=<str>)` sets the destination prompt the goal signal is steered
into. Same registry as notebook `19`, plus one new entry: `intermediate_value`, asking for the
composed query's intermediate value `Fx` instead of the task rule. Pick one by key in
`run_case(..., gs_key=...)`.

In [42]:
from src.goals.goalscope import _build_goalscope_prompt

def _chat(system, user, assistant=""):
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": user},
            {"role": "assistant", "content": assistant}]
    return mt.tokenizer.apply_chat_template(
        msgs, add_generation_prompt=False, continue_final_message=True, tokenize=False)

_ASSIST = "You are a helpful assistant."

GOALSCOPE_PROMPTS = {
    # code-flavoured default (baseline / sanity -- expect it to fit code, not tasks)
    "code_default": _build_goalscope_prompt(mt),
    # function-vector: verbalize the hidden rule / task being performed
    "task_rule":   _chat(_ASSIST, "You are answering a series of questions that all follow the same "
                         "hidden rule. In one short phrase, what task are you performing?"),
    "task_io":     _chat(_ASSIST, "You are mapping each input to an output by a fixed rule. Describe "
                         "the rule in one short phrase: 'I am mapping X to ...'"),
    "task_mixture": _chat(_ASSIST, "You are answering a series of questions that each follow one of a few possible hidden rules. "
                          "In one short phrase, what tasks are you choosing between?"),
    "task_specific": _chat(_ASSIST, "You are answering a series of questions that each follow one of a few possible hidden rules. In one short phrase, what tasks are you choosing between?", "The underlying pattern is mapping X to its"),
    # composing-functions: verbalize the intermediate value Fx (NEW -- everything else here is from notebook 19)
    "intermediate_value": _chat(_ASSIST, "You are answering a series of questions that requires multiple steps of computation to answer. "
                          "In two short phrases, how would you describe the intermediate value, and the second transformation you are performing?"),
    "composition_rule":   _chat(_ASSIST, "You are answering a series of questions that all follow the same composition of hidden "
                             "rules. In a short phrase, what tasks are you composing?"),
    # planning: verbalize the next step
    "next_step":   _chat(_ASSIST, "You are in the middle of writing a response. What is the next step?"),
    "recipe_next": _chat("You are a chef who shares recipes on your blog.",
                         "You are in the middle of writing a recipe. What is the next step?"),
    "trip_next":   _chat("You are a travel expert who helps people plan trips.",
                         "You are in the middle of writing a travel plan. What is the next step?"),
    "story_next":  _chat("You are a short story writer.",
                         "You are in the middle of writing a short story. What should happen next?"),
}
print(list(GOALSCOPE_PROMPTS))

['code_default', 'task_rule', 'task_io', 'task_mixture', 'task_specific', 'intermediate_value', 'composition_rule', 'next_step', 'recipe_next', 'trip_next', 'story_next']


## `run_case` — generate, then scope the generated span

Generates `gen_tokens` from `task_prompt`, then applies GoalScope to **every generated token**
(+ optionally the pivot, the last prompt token, via `include_pivot`). `show=True` (default) renders
the hover-tooltip HTML visualization inline — that's what we actually want to look at.

In [7]:
from src.functional import generate_with_patch
from src.tokens import prepare_input
from src.goals.goalscope import apply_goalscope

@torch.no_grad()
def generate_clean(task_prompt, max_new_tokens=256):
    tok = prepare_input(task_prompt, mt, padding_side="left")
    return generate_with_patch(mt, inputs=tok, n_gen_per_prompt=1,
                               max_new_tokens=max_new_tokens, do_sample=False,
                               remove_prefix=True)[0]

@torch.no_grad()
def run_case(task_prompt, gs_key, *, gen=None, gen_tokens=256,
             strategy="additive_norm", alpha=0.1, steer_layer_range=(35, 46),
             max_new_tokens=48, heads=None, save_path=None, show=True, include_pivot=False):
    """Generate ~gen_tokens from task_prompt, then GoalScope EVERY generated token.
    task_prompt: the prompt to continue (chat OR raw). gs_key: key in GOALSCOPE_PROMPTS.
    include_pivot: also scope the LAST PROMPT TOKEN (pre-answer) -- the point where multiple
    goals may still be entangled, before the generated tokens reveal which one was chosen."""
    heads = heads or goal_heads
    if gen is None:
        gen = generate_clean(task_prompt, gen_tokens)
    rich_print(f"[bold]clean[/bold] [white]{escape(task_prompt)}[/white]"
               f"[sea_green3]{escape(gen)}[/sea_green3]\n")
    enc = prepare_input(task_prompt + gen, mt, padding_side="left")
    p0  = prepare_input(task_prompt, mt, padding_side="left")["input_ids"].shape[1]
    ptot = enc["input_ids"].shape[1]
    start = p0 - 1 if include_pivot else p0
    positions = list(range(start, ptot))                 # ALL generated tokens (+ optionally the pivot)
    results = apply_goalscope(
        prompt=enc, mt=mt, selected_heads=heads,
        strategy=strategy, alpha=alpha, steer_layer_range=steer_layer_range,
        max_new_tokens=max_new_tokens, n_steer_tokens="all",
        goalscope_prompt=GOALSCOPE_PROMPTS[gs_key], token_positions=positions,
        include_noncode=True, title=f"GoalScope · {gs_key} · a={alpha}",
        show=show, save_path=save_path)
    return gen, results

## Composing-functions data (the only new part)

18 tasks across three families (antonym→translation, wikidata two-hop relations,
arithmetic/symbolic); each row gives `x`, `Fx` (first hop), `Gx`, `GFx` (the composed answer),
`FGx`. Prompt format is `"Q: {x}\nA: {GFx}"` -- reuses `fv_common.build_icl_prompt` directly, same
as the FV eval.

In [17]:
import numpy as np
np.unique(ds['task'])

array(['antonym-french', 'antonym-german', 'antonym-spanish',
       'book-author-birthyear', 'landmark-country-capital',
       'mod-twenty-times-two', 'movie-director-birthyear',
       'park-country-capital', 'person-university-founder',
       'person-university-year', 'plus-hundred-times-two',
       'plus-ten-times-two', 'product-company-ceo', 'product-company-hq',
       'rgb-rot120-name', 'song-artist-birthyear', 'word-int-times-two',
       'word-substring-reverse'], dtype='<U25')

In [18]:
from datasets import load_dataset
from scripts.fv.fv_common import build_icl_prompt

PILOT_TASKS = ["antonym-spanish", "book-author-birthyear", "plus-ten-times-two", "landmark-country-capital", "word-substring-reverse"]
N_SHOTS = 10
SEED = 0

ds = load_dataset("apoorvkh/composing-functions")["train"]
by_task = {}
for t in PILOT_TASKS:
    recs = [r for r in ds if r["task"] == t]
    rng = random.Random(SEED)
    rng.shuffle(recs)
    by_task[t] = recs

def to_demos(recs):
    """build_icl_prompt wants {'input': ..., 'output': ...} dicts -- the raw HF records use the
    paper's x/Fx/Gx/GFx/FGx column names, so this is the (previously missing) conversion."""
    return [{"input": r["x"], "output": r["GFx"]} for r in recs]

for t in PILOT_TASKS:
    print(f"{t:28s} n={len(by_task[t])}  e.g. x={by_task[t][0]['x']!r} Fx={by_task[t][0]['Fx']!r} GFx={by_task[t][0]['GFx']!r}")

2026-09-17 16:26:31 httpx INFO     HTTP Request: HEAD https://huggingface.co/datasets/apoorvkh/composing-functions/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-09-17 16:26:31 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/apoorvkh/composing-functions/5b7b70743ff849eae3875dc0c22d5443066b33e9/README.md?%2Fdatasets%2Fapoorvkh%2Fcomposing-functions%2Fresolve%2Fmain%2FREADME.md=&etag=%221bd7a2ec3c2d3817820731c5d41356c706ffeca3%22 "HTTP/1.1 200 OK"
2026-09-17 16:26:31 httpx INFO     HTTP Request: HEAD https://huggingface.co/datasets/apoorvkh/composing-functions/resolve/5b7b70743ff849eae3875dc0c22d5443066b33e9/composing-functions.py "HTTP/1.1 404 Not Found"
2026-09-17 16:26:31 httpx INFO     HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/apoorvkh/composing-functions/apoorvkh/composing-functions.py "HTTP/1.1 404 Not Found"
2026-09-17 16:26:31 httpx INFO     HTTP Request: HEAD https://huggingface.co/dat

## Try it

One query per pilot task, `task_rule` first (sanity check against the FV eval's finding), then
`intermediate_value`. `include_pivot=True` so the query's own `A:` is scoped too, not just the
generated tokens.

In [43]:
task_name = "antonym-spanish"
recs = by_task[task_name]
demos = to_demos(recs[:N_SHOTS])
query = recs[N_SHOTS]
prompt = build_icl_prompt(demos, query["x"])
print(f"x={query['x']!r}  true Fx={query['Fx']!r}  true GFx={query['GFx']!r}")

# gen, _ = run_case(prompt, gs_key="task_rule", gen_tokens=64, alpha=0.1, include_pivot=True)
gen, _ = run_case(prompt, gs_key="composition_rule", gen_tokens=48, alpha=0.1, include_pivot=True)
# gen, _ = run_case(prompt, gs_key="intermediate_value", gen_tokens=64, alpha=0.1, include_pivot=True)

x='indoor'  true Fx='outdoor'  true GFx='al'


clean Q: boom
A: susurro

Q: lead
A: seguir

Q: northern
A: meridional

Q: uphold
A: socavar

Q: fertile
A: infértil

Q: narrow
A: ancho

Q: sound
A: silencio

Q: capable
A: incapaz

Q: playful
A: grave

Q: handsome
A: feo

Q: indoor
A: al aire libre

Q: peaceful
A: agitado

Q: bright
A: oscuro

Q: strong
A: débil

Q: tall
A: bajo

Q: wide
A: estre

GoalScope[additive_norm]:   0%|          | 0/4 [00:00<?, ?it/s]

In [44]:
task_name = "book-author-birthyear"
recs = by_task[task_name]
demos = to_demos(recs[:N_SHOTS])
query = recs[N_SHOTS]
prompt = build_icl_prompt(demos, query["x"])
print(f"x={query['x']!r}  true Fx={query['Fx']!r}  true GFx={query['GFx']!r}")

# gen, _ = run_case(prompt, gs_key="task_rule", gen_tokens=64, alpha=0.1, include_pivot=True)
gen, _ =run_case(prompt, gs_key="composition_rule", gen_tokens=80, alpha=0.1, include_pivot=True, max_new_tokens=64)

x='The Mystery of Marie Rogêt'  true Fx='Edgar Allan Poe'  true GFx='1809'


clean Q: Pyramids
A: 1948

Q: Dirty Hands
A: 1905

Q: Captain Blood
A: 1875

Q: The Idiot
A: 1821

Q: The Village of Stepanchikovo
A: 1821

Q: The Raven
A: 1809

Q: The Turn of the Screw
A: 1843

Q: The Library of Babel
A: 1899

Q: In Search of Lost Time
A: 1871

Q: Tehanu
A: 1929

Q: The Mystery of Marie Rogêt
A: 1842

Q: The Picture of Dorian Gray
A: 1890

Q: The Waste Land
A: 1922

Q: The Great Gatsby
A: 1925

Q: The Catcher in the Rye
A: 1951

Q: To Kill a Mockingbird
A

GoalScope[additive_norm]:   0%|          | 0/6 [00:00<?, ?it/s]

In [36]:
task_name = "plus-ten-times-two"
recs = by_task[task_name]
demos = to_demos(recs[:N_SHOTS])
query = recs[N_SHOTS]
prompt = build_icl_prompt(demos, query["x"])
print(f"x={query['x']!r}  true Fx={query['Fx']!r}  true GFx={query['GFx']!r}")

# gen, _ = run_case(prompt, gs_key="task_rule", gen_tokens=64, alpha=0.1, include_pivot=True)
gen, _ = run_case(prompt, gs_key="intermediate_value", gen_tokens=64, alpha=0.1, include_pivot=True)

x='636'  true Fx='646'  true GFx='1292'


clean Q: 728
A: 1476

Q: 564
A: 1148

Q: 261
A: 542

Q: 47
A: 114

Q: 312
A: 644

Q: 501
A: 1022

Q: 397
A: 814

Q: 839
A: 1698

Q: 150
A: 320

Q: 332
A: 684

Q: 636
A: 1282

Q: 289
A: 598

Q: 421
A: 862

Q: 753
A: 1526

Q: 999
A: 1998

GoalScope[additive_norm]:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
task_name = "landmark-country-capital"
recs = by_task[task_name]
demos = to_demos(recs[:N_SHOTS])
query = recs[N_SHOTS]
prompt = build_icl_prompt(demos, query["x"])
print(f"x={query['x']!r}  true Fx={query['Fx']!r}  true GFx={query['GFx']!r}")

# gen, _ = run_case(prompt, gs_key="task_rule", gen_tokens=64, alpha=0.1, include_pivot=True)
gen, _ = run_case(prompt,gs_key="intermediate_value", gen_tokens=64, alpha=0.1, include_pivot=True, max_new_tokens=64)

x='Serdobsk'  true Fx='Russia'  true GFx='Moscow'


clean Q: Lenzkirch
A: Berlin

Q: Sinan Pasha Mosque
A: Prishtina

Q: Susa
A: Tehran

Q: KV11
A: Cairo

Q: Ephesus
A: Ankara

Q: Oslo Opera House
A: Oslo

Q: Brilon
A: Berlin

Q: Great Ocean Road
A: Canberra

Q: Bodrum Castle
A: Ankara

Q: Smrdáky
A: Bratislava

Q: Serdobsk
A: Moscow

Q: Krasnye Holmy
A: Kyiv

Q: Krasnye Holmy
A: Kyiv

Q: Krasnye Holmy
A: Kyiv

Q: Krasnye Holmy
A: Kyiv

Q: Krasnye

GoalScope[additive_norm]:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
task_name = "word-substring-reverse"
recs = by_task[task_name]
demos = to_demos(recs[:N_SHOTS])
query = recs[N_SHOTS]
prompt = build_icl_prompt(demos, query["x"])
print(f"x={query['x']!r}  true Fx={query['Fx']!r}  true GFx={query['GFx']!r}")

# gen, _ = run_case(prompt, gs_key="task_rule", gen_tokens=64, alpha=0.1, include_pivot=True)
gen, _ = run_case(prompt, gs_key="composition_rule", gen_tokens=64, alpha=0.1, include_pivot=True)
# gen, _ = run_case(prompt, gs_key="intermediate_value", gen_tokens=64, alpha=0.1, include_pivot=True)

x='idiot'  true Fx='idio'  true GFx='oidi'


clean Q: unconditional
A: anoitidnocnu

Q: symptom
A: otpmys

Q: silence
A: cnelis

Q: incorrect
A: cerrocni

Q: triangular
A: alugnairt

Q: insecure
A: rucesni

Q: adoptive
A: vitpoda

Q: outward
A: rawtuo

Q: revive
A: viver

Q: unrestricted
A: etcirtsernu

Q: idiot
A: todi

Q: illusion
A: noilluI

Q: illusionist
A: stnoilluI

Q: illusionary
A: yanoilluI

Q: illusioned
A: dnoilluI

Q: illusioning
A: gnoillu

GoalScope[additive_norm]:   0%|          | 0/5 [00:00<?, ?it/s]

In [39]:
# task_name = "landmark-country-capital"
# recs = by_task[task_name]
# demos = to_demos(recs[:N_SHOTS])
# query = recs[N_SHOTS]
# prompt = build_icl_prompt(demos, query["x"])
# print(f"x={query['x']!r}  true Fx={query['Fx']!r}  true GFx={query['GFx']!r}")

# prompt = "Input: chicken breast, banana, almonds, quinoa, spinach\nOutput: chicken breast, almonds, quinoa, spinach, banana\n\nInput: salmon, white rice, broccoli, peanut butter, apple\nOutput: salmon, peanut butter, broccoli, white rice, apple\n\nInput: eggs, oats, avocado, black beans, orange\nOutput: eggs, black beans, oats, avocado, orange\n\nInput: greek yogurt, sweet potato, walnuts, lentils, strawberries\nOutput: greek yogurt, lentils, walnuts, sweet potato, strawberries\n\nInput: tuna, brown rice, cheese, chickpeas, grapes\nOutput:"
prompt = "Input: turkey, pasta, olive oil, tofu, tomato\nOutput: turkey, tofu, pasta, tomato, olive oil\n\nInput: cod, potato, cashews, kidney beans, pear\nOutput: cod, kidney beans, cashews, potato, pear\n\nInput: cottage cheese, barley, butter, edamame, cucumber\nOutput: cottage cheese, edamame, barley, cucumber, butter\n\nInput: beef, bread, coconut oil, soybeans, lettuce\nOutput: beef, soybeans, bread, lettuce, coconut oil\n\nInput: shrimp, corn, pecans, split peas, watermelon\nOutput:"
# prompt = "Input: duck, amaranth, brazil nuts, garbanzo beans, asparagus\nOutput: duck, garbanzo beans, amaranth, brazil nuts, asparagus\n\nInput: halibut, teff, pine nuts, adzuki beans, cauliflower\nOutput: halibut, adzuki beans, teff, pine nuts, cauliflower\n\nInput: mozzarella, spelt, macadamia nuts, black eyed peas, radish\nOutput: mozzarella, black eyed peas, spelt, macadamia nuts, radish\n\nInput: venison, farro, pistachios, lima beans, onion\nOutput: venison, lima beans, farro, pistachios, onion\n\nInput: lobster, kamut, sesame seeds, cranberry beans, celery\nOutput:"

# gen, _ = run_case(prompt, gs_key="task_rule", gen_tokens=128, alpha=0.1, include_pivot=True)
gen, _ = run_case(prompt,gs_key="composition_rule", gen_tokens=64, alpha=0.1, include_pivot=True, max_new_tokens=128)

clean Input: turkey, pasta, olive oil, tofu, tomato
Output: turkey, tofu, pasta, tomato, olive oil

Input: cod, potato, cashews, kidney beans, pear
Output: cod, kidney beans, cashews, potato, pear

Input: cottage cheese, barley, butter, edamame, cucumber
Output: cottage cheese, edamame, barley, cucumber, butter

Input: beef, bread, coconut oil, soybeans, lettuce
Output: beef, soybeans, bread, lettuce, coconut oil

Input: shrimp, corn, pecans, split peas, watermelon
Output: shrimp, split peas, pecans, corn, watermelon

The pattern seems to be that the output is a rearrangement of the input items, with the primary protein source (if 
any) listed first, followed by other ingredients in no particular order. Is this correct? Yes, your observation is 
correct! The output

GoalScope[additive_norm]:   0%|          | 0/5 [00:00<?, ?it/s]